# 3차 학습

## 1. 환경 설정 + repo clone

In [7]:
# 로컬 환경 설정 (Apple Silicon MPS)
import sys
from pathlib import Path

# 현재 노트북 위치 기준으로 repo 루트를 sys.path에 추가
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
print(f"repo 루트: {ROOT}")

import torch
print(f"torch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
print(f"MPS:  {torch.backends.mps.is_available()}")

# Apple Silicon MPS 우선, 없으면 CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"device: {device}")

# 데이터 파일 확인
data_path = ROOT / "data" / "nemotron_100k_seed42.parquet"
assert data_path.exists(), f"데이터 파일 없음: {data_path}"
print(f"✓ 데이터: {data_path} ({data_path.stat().st_size / 1024 / 1024:.1f} MB)")

repo 루트: /Users/irene.msp/park-personal/DL-teamproject
torch: 2.12.0
CUDA: False
MPS:  True
device: mps
✓ 데이터: /Users/irene.msp/park-personal/DL-teamproject/data/nemotron_100k_seed42.parquet (190.7 MB)


## 2. 데이터 로드 + 라벨 + 분할 (src 모듈 사용)

In [8]:
from src.data_utils import load_and_label, stratified_split, build_geo_pattern, build_input

df, meta = load_and_label(ROOT / "data" / "nemotron_100k_seed42.parquet")
train_df, val_df, test_df = stratified_split(df, seed=42)
print(f"train: {len(train_df):,}  val: {len(val_df):,}  test: {len(test_df):,}")

geo_pattern, geo_terms = build_geo_pattern(df, meta["prov_list"])
print(f"지명 마스킹 사전: {len(geo_terms)} 항목")

원본 shape: (100000, 26)
라이프 매칭 실패 drop: 12079 → (87921, 31)
train: 70,336  val: 8,792  test: 8,793
지명 마스킹 사전: 474 항목


## 3. 토크나이저 빌드 + 3 condition 인코딩

In [10]:
from src.tokenizer import SyllableTokenizer
from tqdm.auto import tqdm

MAX_LEN = 512

# vocab은 train의 raw에서만
train_raw_texts = [build_input(row, geo_pattern, "raw") for _, row in tqdm(train_df.iterrows(),
                                                                            total=len(train_df), desc="train raw text")]
tokenizer = SyllableTokenizer.build(train_raw_texts, max_len=MAX_LEN)
print(f"Vocab: {tokenizer.vocab_size:,}")

/opt/anaconda3/envs/dl-demo/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
train raw text: 100%|██████████| 70336/70336 [00:09<00:00, 7469.86it/s]


Vocab: 1,590


In [11]:
# 3 condition × 3 split 인코딩 캐시
token_cache = {}
for cond in ["raw", "masked", "masked_geo"]:
    for split_name, sdf in [("train", train_df), ("val", val_df), ("test", test_df)]:
        if cond == "raw" and split_name == "train":
            texts = train_raw_texts  # 재사용
        else:
            texts = [build_input(row, geo_pattern, cond) for _, row in tqdm(
                sdf.iterrows(), total=len(sdf), desc=f"{cond}/{split_name}", leave=False)]
        token_cache[(cond, split_name)] = tokenizer.encode_batch(texts)
print(f"캐시 완료: {len(token_cache)} (3 cond × 3 split)")

캐시 완료: 9 (3 cond × 3 split)


## 4. 학습 준비 — Loaders + Loss + Class weights

In [12]:
from src.train import compute_class_weights, make_loaders

class_weights = compute_class_weights(train_df, meta["task_config"], meta["task_names"], device)
print("Class weights 계산 완료")

def get_loaders(condition, batch_size=512):
    return make_loaders(
        token_cache[(condition, "train")],
        token_cache[(condition, "val")],
        token_cache[(condition, "test")],
        train_df, val_df, test_df,
        meta["task_config"], meta["task_names"],
        batch_size=batch_size,
    )

Class weights 계산 완료


## 5. 모델 저장 경로 + meta.json 

In [14]:
import os, json
import shutil

MODEL_DIR = ROOT / "models"
RESULTS_DIR = ROOT / "results"
MODEL_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)
print(f"✓ 폴더 준비: {MODEL_DIR}, {RESULTS_DIR}")

# meta.json 저장 (vocab + label_maps + geo_terms 모두)
meta_save = {
    "vocab_size": tokenizer.vocab_size,
    "max_len": tokenizer.max_len,
    "pad_idx": 0, "unk_idx": 1,
    "itos": tokenizer.itos,
    "task_names": meta["task_names"],
    "task_config": meta["task_config"],
    "label_maps": {
        "sex": ["여성", "남성"],
        "age": meta["age_labels"],
        "prov": meta["prov_list"],
        "life": meta["life_labels"],
        "marital": meta["marital_labels"],
    },
    "geo_terms": geo_terms,
}
with open(MODEL_DIR / "meta.json", "w", encoding="utf-8") as f:
    json.dump(meta_save, f, ensure_ascii=False, indent=2)
print(f"✓ meta.json 저장 (vocab {tokenizer.vocab_size:,}, geo {len(geo_terms):,})")

def save_model(model, name):
    path = MODEL_DIR / f"{name}.pt"
    torch.save(model.state_dict(), path)
    n_params = sum(x.numel() for x in model.parameters())
    print(f"✓ {name}.pt 저장 ({n_params/1e3:.0f}k params)")

✓ 폴더 준비: /Users/irene.msp/park-personal/DL-teamproject/models, /Users/irene.msp/park-personal/DL-teamproject/results
✓ meta.json 저장 (vocab 1,590, geo 474)


## 6. 3 모델 학습 (TextCNN / BiLSTM / CNN-LSTM)

각 ~5분.

In [15]:
from src.encoders import TextCNN, BiLSTM, CNNLSTM
from src.train import train_model

vocab_size = tokenizer.vocab_size
task_config = meta["task_config"]
task_names = meta["task_names"]

# [1/4] TextCNN
print("="*60); print("[1/4] TextCNN 5-task raw"); print("="*60)
tr, va, te = get_loaders("raw")
textcnn, info1 = train_model(TextCNN, tr, va, te, vocab_size, task_config,
                              task_names, class_weights, device, n_epochs=10)
save_model(textcnn, "textcnn_5task")

[1/4] TextCNN 5-task raw


/opt/anaconda3/envs/dl-demo/lib/python3.10/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


  ep  1  val avg-F1 0.4885  prov 0.0463


/opt/anaconda3/envs/dl-demo/lib/python3.10/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


  ep  2  val avg-F1 0.5553  prov 0.0776


...

  ep  9  val avg-F1 0.7554  prov 0.6192


/opt/anaconda3/envs/dl-demo/lib/python3.10/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


  ep 10  val avg-F1 0.7699  prov 0.6953


/opt/anaconda3/envs/dl-demo/lib/python3.10/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


  → test avg 0.7672  prov 0.6923
✓ textcnn_5task.pt 저장 (595k params)


In [16]:
# [2/4] BiLSTM
print("="*60); print("[2/4] BiLSTM 5-task raw"); print("="*60)
tr, va, te = get_loaders("raw")
bilstm, info2 = train_model(BiLSTM, tr, va, te, vocab_size, task_config,
                             task_names, class_weights, device, n_epochs=10)
save_model(bilstm, "bilstm_5task")

[2/4] BiLSTM 5-task raw


/opt/anaconda3/envs/dl-demo/lib/python3.10/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


  ep  1  val avg-F1 0.2668  prov 0.0151


/opt/anaconda3/envs/dl-demo/lib/python3.10/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


  ep  2  val avg-F1 0.3960  prov 0.0307


...

  ep  9  val avg-F1 0.6144  prov 0.0850


/opt/anaconda3/envs/dl-demo/lib/python3.10/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


  ep 10  val avg-F1 0.6327  prov 0.1849


/opt/anaconda3/envs/dl-demo/lib/python3.10/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


  → test avg 0.6286  prov 0.1811
✓ bilstm_5task.pt 저장 (703k params)


In [17]:
# [3/4] CNN-LSTM
print("="*60); print("[3/4] CNN-LSTM 5-task raw"); print("="*60)
tr, va, te = get_loaders("raw")
cnnlstm, info3 = train_model(CNNLSTM, tr, va, te, vocab_size, task_config,
                              task_names, class_weights, device, n_epochs=10)
save_model(cnnlstm, "cnnlstm_5task")

[3/4] CNN-LSTM 5-task raw


/opt/anaconda3/envs/dl-demo/lib/python3.10/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


  ep  1  val avg-F1 0.1160  prov 0.0059


/opt/anaconda3/envs/dl-demo/lib/python3.10/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


  ep  2  val avg-F1 0.0865  prov 0.0188


...

  ep  3  val avg-F1 0.0872  prov 0.0131


/opt/anaconda3/envs/dl-demo/lib/python3.10/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


  ep  4  val avg-F1 0.0898  prov 0.0260
  early stop @ ep 4


/opt/anaconda3/envs/dl-demo/lib/python3.10/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


  → test avg 0.1162  prov 0.0063
✓ cnnlstm_5task.pt 저장 (732k params)


## 7. TextCNN Mix-training 

raw + masked_geo 50:50 혼합 학습.  
masked_geo 조건에서 광역시도 F1이 얼마나 올라가는지 확인.

In [18]:
from src.mix_training import MixedConditionDataset, make_mix_loader
from src.encoders import PersonaModel
from src.train import UncertaintyWeightedLoss, evaluate
import torch.nn as nn

encoded_variants = {
    "raw":        token_cache[("raw", "train")],
    "masked_geo": token_cache[("masked_geo", "train")],
}
mix_loader = make_mix_loader(encoded_variants, train_df, task_config, task_names,
                              mix_probs={"raw": 0.5, "masked_geo": 0.5}, batch_size=512)
_, va, _ = get_loaders("raw")    # validation은 raw

print("="*60); print("[4/4] TextCNN Mix-training (raw+masked_geo 50:50)"); print("="*60)

model = PersonaModel(TextCNN(vocab_size=vocab_size), task_config).to(device)
loss_fn = UncertaintyWeightedLoss(task_names, class_weights).to(device)
optimizer = torch.optim.AdamW(
    list(model.parameters()) + list(loss_fn.parameters()),
    lr=1e-3, weight_decay=1e-5)

best_f1, best_state, pc = -1, None, 0
for epoch in range(1, 11):
    model.train()
    for batch in mix_loader:
        x = batch[0].to(device)
        labels = {n: batch[i+1].to(device) for i, n in enumerate(task_names)}
        optimizer.zero_grad()
        loss, _ = loss_fn(model(x), labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
    val_m = evaluate(model, va, task_names, device)
    print(f"  ep {epoch:2d}  val avg-F1 {val_m['avg_f1']:.4f}  prov {val_m['prov_f1']:.4f}")
    if val_m["avg_f1"] > best_f1:
        best_f1 = val_m["avg_f1"]
        best_state = {k: v.clone().cpu() for k, v in model.state_dict().items()}
        pc = 0
    else:
        pc += 1
        if pc >= 3:
            print(f"  early stop @ ep {epoch}"); break

model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

print("\n[Mix 모델 test 평가 — 3 condition 비교]")
for cond in ["raw", "masked", "masked_geo"]:
    _, _, te = get_loaders(cond)
    m = evaluate(model, te, task_names, device)
    print(f"  {cond:12s}: avg-F1 {m['avg_f1']:.4f}  prov-F1 {m['prov_f1']:.4f}")

save_model(model, "textcnn_mix")

[4/4] TextCNN Mix-training (raw+masked_geo 50:50)


/opt/anaconda3/envs/dl-demo/lib/python3.10/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


  ep  1  val avg-F1 0.4473  prov 0.0385


/opt/anaconda3/envs/dl-demo/lib/python3.10/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


  ep  2  val avg-F1 0.5503  prov 0.0270


...

  raw         : avg-F1 0.6784  prov-F1 0.2271


/opt/anaconda3/envs/dl-demo/lib/python3.10/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


  masked      : avg-F1 0.6342  prov-F1 0.2127


/opt/anaconda3/envs/dl-demo/lib/python3.10/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


  masked_geo  : avg-F1 0.6242  prov-F1 0.1519
✓ textcnn_mix.pt 저장 (595k params)


## 8. 30 페르소나 선별 → personas_30.json

데모에서 사용할 페르소나 30개를 test set에서 stereotype 단서 풍부한 것 위주로 선별.

In [19]:
import random

STEREOTYPE_KEYWORDS = {
    "age": {"트로트":"고령층","임영웅":"고령층","막걸리":"고령층","손주":"고령층","은퇴":"고령층",
            "e스포츠":"청년층","PC방":"청년층","유튜브":"청년층","스타트업":"청년층"},
    "prov": {"갯벌":"전라남","바닷바람":"해안","무등산":"광주","한라산":"제주","감귤":"제주",
             "해운대":"부산","돼지국밥":"부산"},
    "sex": {"축구":"남성","골프":"남성","육아":"여성","살림":"여성","베이킹":"여성","요리":"여성"},
    "marital": {"손주":"기혼","자녀":"기혼","아이와":"기혼"},
}
TASK_KR = {"sex":"성별","age":"연령대","prov":"광역시도","life":"라이프스타일","marital":"결혼여부"}

def find_cues(text):
    cues = []
    for task, kws in STEREOTYPE_KEYWORDS.items():
        for kw, imp in kws.items():
            if kw in text:
                cues.append({"word": kw, "task": TASK_KR[task], "implies": imp})
    return cues

random.seed(42)
cands = []
for _, row in test_df.iterrows():
    raw = build_input(row, geo_pattern, "raw")
    cands.append((row, len(find_cues(raw))))
cands.sort(key=lambda x: -x[1])
pool = cands[:80]
random.shuffle(pool)
selected = [r for r, _ in pool[:30]]
print(f"30 페르소나 선별 완료")

30 페르소나 선별 완료


In [20]:
import torch.nn.functional as F

# 각 페르소나에 대해 3 모델 × 3 condition 추론
LABEL_MAPS = meta_save["label_maps"]

def predict_for_demo(model, row, cond):
    text = build_input(row, geo_pattern, cond)
    ids = tokenizer.encode_batch([text]).to(device)
    model.eval()
    with torch.no_grad():
        logits = model(ids)
    out = {}
    for task in task_names:
        prob = F.softmax(logits[task], dim=1)[0]
        idx = prob.argmax().item()
        true_idx = int(row[task_config[task]["col"]])
        out[task] = {
            "pred": LABEL_MAPS[task][idx],
            "conf": round(float(prob[idx]), 3),
            "true": LABEL_MAPS[task][true_idx],
            "correct": bool(idx == true_idx),
        }
    return out

demo_models = {"TextCNN": textcnn, "BiLSTM": bilstm, "CNNLSTM": cnnlstm}

personas_out = {"tasks": task_names, "task_kr": TASK_KR, "personas": []}
for i, row in enumerate(selected):
    raw_text = build_input(row, geo_pattern, "raw")
    entry = {
        "id": i,
        "label": f"{LABEL_MAPS['sex'][int(row['y_sex'])]} {LABEL_MAPS['age'][int(row['y_age'])]} · {LABEL_MAPS['prov'][int(row['y_prov'])]}",
        "texts": {c: build_input(row, geo_pattern, c)[:300] for c in ["raw","masked","masked_geo"]},
        "cues": find_cues(raw_text),
        "models": {},
    }
    for mname, model in demo_models.items():
        entry["models"][mname] = {c: predict_for_demo(model, row, c) for c in ["raw","masked","masked_geo"]}
    personas_out["personas"].append(entry)

# results/ 폴더에 저장 (로컬)
out_path = RESULTS_DIR / "personas_30.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(personas_out, f, ensure_ascii=False, indent=2)
print(f"✓ personas_30.json 저장 → {out_path}")
print(f"  ({len(personas_out['personas'])} 페르소나 × 3 모델 × 3 condition)")

✓ personas_30.json 저장 → /Users/irene.msp/park-personal/DL-teamproject/results/personas_30.json
  (30 페르소나 × 3 모델 × 3 condition)
